# ACDA-PatternGAN — Aesthetic-Controllable Disentangled Apparel Pattern GAN
### Full proposed model + ablation study on **Batik Lasem** (64×64)

**Research:** *Deep GANs for Aesthetic-Driven Apparel Pattern Synthesis and Interactive Visualization.*

This single notebook implements FOUR novelties and a fair A→F ablation ladder:
1. **Multi-Attribute Aesthetic Conditioning** — `c = [style, color, complexity, density, symmetry]`
2. **Disentangled Latent Representation** — `z_structure(48) | z_style(32) | z_color(24) | z_texture(24)` (128)
3. **Seamless / Repeat-Aware synthesis** — differentiable periodic-boundary loss `L_SEAM`
4. **Joint Aesthetic–Novelty Optimization** — `L_AEST`, `L_NOV`

Every model (A–F) is evaluated with the SAME fixed 9-metric suite: **FID, KID, LPIPS, Diversity,
Novelty, Computational Aesthetic Score, Seamless Score, Condition Adherence, MMD**.

> **Scientific-integrity rules honoured in code:** no metric value is hard-coded or fabricated;
> the TEST set is used ONLY for final reporting (never for model selection / early-stopping / LR /
> tuning); best checkpoint is chosen by **validation** FID; novelty is measured against **training**
> data only; targets are goals, not results ("Target not achieved" is reported honestly).

**Ablation ladder** (each step adds exactly one component):
`A baseline → B +conditioning → C +disentanglement → D +seamless → E +aesthetic → F +novelty (=Full)`.
(Novelty 4 "joint aesthetic–novelty" is split across E and F so each rung adds one term.)

Configure only `DATASET_ROOT` and `OUTPUT_ROOT` in cell 02, then run top-to-bottom.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 02 — Configuration (edit only DATASET_ROOT / OUTPUT_ROOT)

In [8]:

# The ONLY things you normally edit:
DATASET_ROOT   = "/content/drive/MyDrive/hari/Textile_Pattern_GAN/Datasets"   # folder containing Batik_Lasem/
OUTPUT_ROOT    = "/content/drive/MyDrive/hari/Textile_Pattern_GAN/acda_outputs"
CHECKPOINT_ROOT = "OUTPUT_ROOT/checkpoints"   # None -> OUTPUT_ROOT/checkpoints

# ---- Everything below has sensible defaults ----
CFG = dict(
    seed=42,
    image_size=64,
    channels=3,
    # latent (Novelty 2) — sums to latent_dim
    latent_dim=128, z_structure=48, z_style=32, z_color=24, z_texture=24,
    # training
    batch_size=64, epochs=60, lr_g=2e-4, lr_d=2e-4, beta1=0.5, beta2=0.999,
    n_critic=1, use_amp=True, num_workers=2,
    # loss weights
    lambda_aest=0.5, lambda_dis=1.0, lambda_seam=1.0, lambda_nov=0.2,
    # evaluation (fixed subsets for epoch-wise metrics; full protocol at final test)
    eval_subset=256, eval_n_gen=256, kid_subset=50, final_n_gen=1156,
    # early stopping / scheduler (VALIDATION only)
    es_monitor="val_FID", es_mode="min", es_patience=15,
    lr_patience=6, lr_factor=0.5, min_lr=1e-6,
    # split (used only if no existing split files are found)
    val_frac=0.10, test_frac=0.20,
    # research targets (GOALS, not results)
    targets=dict(FID=80.0, KID=0.05, AESTHETIC=0.60, SEAMLESS=0.80, NOVELTY=0.30),
    # composite validation-score weights (transparent, configurable)
    composite_weights=dict(FID=0.30, KID=0.15, MMD=0.10, diversity=0.10,
                           novelty=0.10, aesthetic=0.10, seamless=0.075, condition_adherence=0.075),
)
MOTIF_ORDER = ["Gunung Ringgit", "Kricak_Watu Pecah", "Latohan", "Nyuk Pitu", "Seritan"]
assert CFG["z_structure"]+CFG["z_style"]+CFG["z_color"]+CFG["z_texture"] == CFG["latent_dim"]
print("Config OK. image_size=%d latent_dim=%d epochs=%d" % (CFG["image_size"], CFG["latent_dim"], CFG["epochs"]))

Config OK. image_size=64 latent_dim=128 epochs=60


## 03 — Environment installation

In [9]:

import sys, subprocess
def pip(*pkgs):
    subprocess.run([sys.executable,"-m","pip","install","-q",*pkgs], check=False)
# torch/torchvision preinstalled on Colab. Add metric deps.
pip("torchmetrics>=1.0.0", "torch-fidelity", "lpips", "scipy", "scikit-learn", "pandas", "matplotlib", "pillow")
print("Environment install step done.")

Environment install step done.


## 04 — Imports

In [10]:

import os, json, time, math, random, hashlib, glob, warnings
from collections import OrderedDict, Counter
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
warnings.filterwarnings("ignore")
print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| cuda", torch.version.cuda)

torch 2.11.0+cu128 | torchvision 0.26.0+cu128 | cuda 12.8


## 05 — Reproducibility

In [11]:

def set_seed(seed=CFG["seed"], deterministic=True):
    os.environ["PYTHONHASHSEED"]=str(seed)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic
    print("SEED =", seed)
set_seed()
def seed_worker(_):
    ws = torch.initial_seed() % 2**32
    np.random.seed(ws); random.seed(ws)
G_SEED = torch.Generator(); G_SEED.manual_seed(CFG["seed"])

SEED = 42


## 06 — GPU verification

In [12]:

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("="*60); print("ENVIRONMENT"); print("="*60)
print("PyTorch    :", torch.__version__)
print("CUDA       :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU        :", p.name)
    print("GPU Memory :", round(p.total_memory/1024**3,2), "GB")
else:
    print("WARNING: no GPU. On Colab: Runtime > Change runtime type > GPU (L4).")
print("Device     :", DEVICE); print("="*60)
def gpu_mem_mb():
    return round(torch.cuda.max_memory_allocated()/1024**2,1) if torch.cuda.is_available() else 0.0

ENVIRONMENT
PyTorch    : 2.11.0+cu128
CUDA       : 12.8 | available: True
GPU        : NVIDIA L4
GPU Memory : 22.03 GB
Device     : cuda


## 07 — Google Drive + path verification

In [13]:

try:
    from google.colab import drive; drive.mount("/content/drive"); IN_COLAB=True
except Exception as e:
    IN_COLAB=False; print("Not in Colab / already mounted:", e)

CHECKPOINT_ROOT = CHECKPOINT_ROOT or os.path.join(OUTPUT_ROOT, "checkpoints")
SUBDIRS = ["dataset_audit","checkpoints","histories","metrics","plots","samples",
           "traversals","ablations","final_results"]
for d in SUBDIRS:
    os.makedirs(os.path.join(OUTPUT_ROOT, d), exist_ok=True)

BATIK_DIR = os.path.join(DATASET_ROOT, "Batik_Lasem")
if not os.path.isdir(BATIK_DIR):
    raise FileNotFoundError(
        "Batik_Lasem not found under DATASET_ROOT.\nExpected: %s\n"
        "Edit DATASET_ROOT in cell 02." % BATIK_DIR)
print("DATASET_ROOT OK :", DATASET_ROOT)
print("OUTPUT_ROOT     :", OUTPUT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATASET_ROOT OK : /content/drive/MyDrive/hari/Textile_Pattern_GAN/Datasets
OUTPUT_ROOT     : /content/drive/MyDrive/hari/Textile_Pattern_GAN/acda_outputs


## 08 — Dataset discovery (locate metadata + canonical images)

In [14]:

# Locate the Batik_Lasem metadata CSV (semicolon-delimited, UTF-8 BOM).
def find_metadata(root):
    for pat in ["metadata motifs.csv","metadata_motifs.csv","*etadata*.csv"]:
        hits = glob.glob(os.path.join(root, "**", pat), recursive=True)
        if hits: return sorted(hits, key=len)[0]
    return None
METADATA_PATH = find_metadata(BATIK_DIR)
if METADATA_PATH is None:
    raise FileNotFoundError("Could not find Batik_Lasem 'metadata motifs.csv' under "+BATIK_DIR)
print("Metadata:", METADATA_PATH)

meta = pd.read_csv(METADATA_PATH, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
meta = meta.loc[:, [c for c in meta.columns if not c.startswith("Unnamed") and c.strip()]]
meta.columns = [c.strip() for c in meta.columns]
for c in meta.columns: meta[c] = meta[c].astype(str).str.strip()
print("Metadata rows:", len(meta), "| columns:", list(meta.columns))

# canonical image = the file in the motif's `all_motifs (N)` folder (native res, one per instance)
MOTIF_FOLDER = {"Gunung Ringgit":"Gunung Ringgit","Kricak / Watu Pecah":"Kricak_Watu Pecah",
                "Latohan":"Latohan","Nyuk Pitu":"Nyuk Pitu","Seritan":"Seritan"}
_counts = meta["motif_name"].value_counts().to_dict()
def resolve_path(filename, motif):
    folder = MOTIF_FOLDER.get(motif, motif)
    cand = os.path.join(BATIK_DIR, "motifs (isen-isen)", folder, "all_motifs (%d)"%_counts.get(motif,0), filename)
    if os.path.isfile(cand): return cand
    hits = glob.glob(os.path.join(BATIK_DIR,"**","all_motifs*",filename), recursive=True)
    if hits: return hits[0]
    hits = glob.glob(os.path.join(BATIK_DIR,"**",filename), recursive=True)
    return hits[0] if hits else None

# verified motif categories (read from metadata, not assumed)
DATA_MOTIFS = sorted(meta["motif_name"].unique().tolist())
print("Motif categories in metadata:", DATA_MOTIFS)
MOTIFS = [m for m in MOTIF_ORDER if m in DATA_MOTIFS] + [m for m in DATA_MOTIFS if m not in MOTIF_ORDER]
MOTIF_TO_IDX = {m:i for i,m in enumerate(MOTIFS)}
NUM_MOTIFS = len(MOTIFS)
print("Using %d motif classes:" % NUM_MOTIFS, MOTIFS)

Metadata: /content/drive/MyDrive/hari/Textile_Pattern_GAN/Datasets/Batik_Lasem/motifs (isen-isen)/metadata motifs.csv
Metadata rows: 5860 | columns: ['filename', 'motif_name', 'origin_images', 'Origin_images_Type', 'Source Workshop', 'year_collected', 'image_type', 'image_resolution', 'color_profile', 'annotator_initials']
Motif categories in metadata: ['Gunung Ringgit', 'Kricak / Watu Pecah', 'Latohan', 'Nyuk Pitu', 'Seritan']
Using 5 motif classes: ['Gunung Ringgit', 'Latohan', 'Nyuk Pitu', 'Seritan', 'Kricak / Watu Pecah']


## 09 — Dataset audit (counts, dims, channels, duplicates)

In [15]:

canon = meta.copy()
canon["path"] = [resolve_path(f, m) for f, m in zip(canon["filename"], canon["motif_name"])]
n_missing = int(canon["path"].isna().sum())
canon = canon[canon["path"].notna()].reset_index(drop=True)
print("Canonical instances (metadata):", len(meta), "| resolved on disk:", len(canon),
      "| missing:", n_missing)

# sample audit: dims / channels / readability / md5 duplicates
rng = np.random.default_rng(CFG["seed"])
probe_idx = rng.choice(len(canon), size=min(500,len(canon)), replace=False)
dims=Counter(); modes=Counter(); unreadable=0; md5=Counter()
for i in probe_idx:
    p = canon["path"].iloc[int(i)]
    try:
        im=Image.open(p); im.verify(); im=Image.open(p)
        dims[im.size]+=1; modes[im.mode]+=1
        md5[hashlib.md5(open(p,"rb").read()).hexdigest()]+=1
    except Exception:
        unreadable+=1
audit = dict(metadata_rows=int(len(meta)), resolved=int(len(canon)), missing=n_missing,
             probe=len(probe_idx), unreadable=unreadable,
             dims=dict((f"{w}x{h}",c) for (w,h),c in dims.items()),
             modes=dict(modes),
             exact_dup_groups_in_probe=int(sum(1 for v in md5.values() if v>1)),
             motif_distribution=canon["motif_name"].value_counts().to_dict())
os.makedirs(os.path.join(OUTPUT_ROOT,"dataset_audit"), exist_ok=True)
json.dump(audit, open(os.path.join(OUTPUT_ROOT,"dataset_audit","audit.json"),"w"), indent=2)
print(json.dumps(audit, indent=2))
if unreadable>0: print("WARNING: unreadable images encountered in probe.")

Canonical instances (metadata): 5860 | resolved on disk: 5860 | missing: 0
{
  "metadata_rows": 5860,
  "resolved": 5860,
  "missing": 0,
  "probe": 500,
  "unreadable": 0,
  "dims": {
    "28x28": 436,
    "142x142": 64
  },
  "modes": {
    "RGB": 500
  },
  "exact_dup_groups_in_probe": 0,
  "motif_distribution": {
    "Latohan": 1373,
    "Seritan": 1328,
    "Nyuk Pitu": 1291,
    "Kricak / Watu Pecah": 1004,
    "Gunung Ringgit": 864
  }
}


## 10 — Train/Val/Test split verification (group-aware by origin_images)

In [16]:

# 1) Try to reuse EXISTING split files (Phase-1/2) if present.
SPLIT_DIR = os.path.join(OUTPUT_ROOT, "dataset_audit")
CAND_DIRS = [SPLIT_DIR, os.path.join(DATASET_ROOT,"splits"),
             os.path.join(BATIK_DIR,"splits"), os.path.join(OUTPUT_ROOT,"splits")]
def _find(name):
    for d in CAND_DIRS:
        for pat in [name+".csv", "batik_lasem_"+name+".csv", "*"+name+"*.csv"]:
            hits = glob.glob(os.path.join(d, pat))
            if hits: return hits[0]
    return None
tr_f, va_f, te_f = _find("train"), _find("val"), _find("test")

def _load(fp):
    df = pd.read_csv(fp)
    fcol = "filename" if "filename" in df.columns else df.columns[0]
    df = df.rename(columns={fcol:"filename"})
    return df.merge(canon[["filename","motif_name","origin_images","path"]], on="filename", how="left",
                    suffixes=("","_c"))

if tr_f and va_f and te_f:
    print("Using EXISTING split files:\n ", tr_f, "\n ", va_f, "\n ", te_f)
    train_df, val_df, test_df = _load(tr_f), _load(va_f), _load(te_f)
else:
    print("No existing 3-way split found -> building a deterministic group-aware split (seed=%d)." % CFG["seed"])
    # group-aware by origin_images; randomized search keeps motif distribution close and all motifs present
    origins = list(canon.groupby("origin_images"))
    names = [o for o,_ in origins]
    vecs  = np.array([[int((g.motif_name==m).sum()) for m in MOTIFS] for _,g in origins], float)
    sizes = vecs.sum(1); total=len(canon); motif_tot=vecs.sum(0)
    rng2 = np.random.default_rng(CFG["seed"])
    def gaps(val_mask, test_mask):
        tr = motif_tot - vecs[val_mask].sum(0) - vecs[test_mask].sum(0)
        va = vecs[val_mask].sum(0); te = vecs[test_mask].sum(0)
        if min(tr.sum(),va.sum(),te.sum())==0 or (tr==0).any() or (va==0).any() or (te==0).any():
            return 1e9
        f=lambda v:100*v/v.sum()
        return float(np.max(np.abs(f(tr)-f(va))) + np.max(np.abs(f(tr)-f(te))))
    best=None
    for _ in range(30000):
        perm=rng2.permutation(len(names)); va=np.zeros(len(names),bool); te=np.zeros(len(names),bool)
        tot=0
        for i in perm:
            if tot < CFG["test_frac"]*total: te[i]=True; tot+=sizes[i]
            else: break
        tot=0
        for i in perm[::-1]:
            if te[i]: continue
            if tot < CFG["val_frac"]*total: va[i]=True; tot+=sizes[i]
            else: break
        g=gaps(va,te)
        if best is None or g<best[0]: best=(g,va.copy(),te.copy())
    _,va_mask,te_mask=best
    assign={}
    for i,nm in enumerate(names):
        assign[nm] = "test" if te_mask[i] else ("val" if va_mask[i] else "train")
    canon["split"]=canon["origin_images"].map(assign)
    train_df=canon[canon.split=="train"].copy(); val_df=canon[canon.split=="val"].copy(); test_df=canon[canon.split=="test"].copy()
    for nm,df in [("train",train_df),("val",val_df),("test",test_df)]:
        df.drop(columns=["split"],errors="ignore").to_csv(os.path.join(SPLIT_DIR,"batik_lasem_%s.csv"%nm), index=False)
    print("Saved split CSVs to", SPLIT_DIR, "(max motif%% gap=%.2f)"%best[0])

# ---- verify + report ----
def dist(df): return {m:int((df.motif_name==m).sum()) for m in MOTIFS}
sizes = dict(train=len(train_df), val=len(val_df), test=len(test_df))
overlap = dict(
    train_val=len(set(train_df.origin_images)&set(val_df.origin_images)),
    train_test=len(set(train_df.origin_images)&set(test_df.origin_images)),
    val_test=len(set(val_df.origin_images)&set(test_df.origin_images)),
    fn_train_val=len(set(train_df.filename)&set(val_df.filename)),
    fn_train_test=len(set(train_df.filename)&set(test_df.filename)),
    fn_val_test=len(set(val_df.filename)&set(test_df.filename)))
print("Split sizes :", sizes, "| total:", sum(sizes.values()))
print("Reference (approx.): train~4115 val~589 test~1156 (actual computed above are authoritative)")
print("Train dist:", dist(train_df)); print("Val dist  :", dist(val_df)); print("Test dist :", dist(test_df))
print("Origin overlap (must be 0 for group-aware):", {k:v for k,v in overlap.items() if not k.startswith("fn")})
print("Filename overlap (must be 0):", {k:v for k,v in overlap.items() if k.startswith("fn")})
leak = any(v>0 for v in overlap.values())
if leak: print("WARNING: split overlap detected!  (existing split files may be image-level, not group-aware)")
json.dump(dict(sizes=sizes, overlap=overlap, train=dist(train_df), val=dist(val_df), test=dist(test_df)),
          open(os.path.join(SPLIT_DIR,"split_report.json"),"w"), indent=2)

No existing 3-way split found -> building a deterministic group-aware split (seed=42).
Saved split CSVs to /content/drive/MyDrive/hari/Textile_Pattern_GAN/acda_outputs/dataset_audit (max motif% gap=14.36)
Split sizes : {'train': 3990, 'val': 590, 'test': 1280} | total: 5860
Reference (approx.): train~4115 val~589 test~1156 (actual computed above are authoritative)
Train dist: {'Gunung Ringgit': 626, 'Latohan': 904, 'Nyuk Pitu': 865, 'Seritan': 937, 'Kricak / Watu Pecah': 658}
Val dist  : {'Gunung Ringgit': 121, 'Latohan': 141, 'Nyuk Pitu': 174, 'Seritan': 98, 'Kricak / Watu Pecah': 56}
Test dist : {'Gunung Ringgit': 117, 'Latohan': 328, 'Nyuk Pitu': 252, 'Seritan': 293, 'Kricak / Watu Pecah': 290}
Origin overlap (must be 0 for group-aware): {'train_val': 0, 'train_test': 0, 'val_test': 0}
Filename overlap (must be 0): {'fn_train_val': 0, 'fn_train_test': 0, 'fn_val_test': 0}


## 11 — Metadata preparation for conditioning

In [17]:

# style = motif index; the 4 continuous aesthetic attributes are COMPUTED from pixels in cell 12.
for df in (train_df, val_df, test_df):
    df["motif_idx"] = df["motif_name"].map(MOTIF_TO_IDX).astype(int)
print("Assigned motif_idx. Example:", train_df[["filename","motif_name","motif_idx"]].head(3).to_dict("records"))

Assigned motif_idx. Example: [{'filename': 'GunungRinggit_00001.jpg', 'motif_name': 'Gunung Ringgit', 'motif_idx': 0}, {'filename': 'GunungRinggit_00002.jpg', 'motif_name': 'Gunung Ringgit', 'motif_idx': 0}, {'filename': 'GunungRinggit_00003.jpg', 'motif_name': 'Gunung Ringgit', 'motif_idx': 0}]


## 12 — Objective aesthetic-attribute functions (differentiable, torch)
These compute the 4 continuous condition attributes **from pixels** — used both as conditioning
targets and to *measure* condition adherence / aesthetic score. All are objective descriptors in
[0,1]; none are human ratings. Same functions used for losses (with grad) and metrics (detached).

In [18]:

def _to_bchw(x):
    if x.dim()==3: x=x.unsqueeze(0)
    return x
def img01(x):  # accept [-1,1] or [0,1] -> [0,1]
    x=_to_bchw(x).float()
    if float(x.min())<-0.01: x=x*0.5+0.5
    return x.clamp(0,1)
def luminance(x01):  # (B,1,H,W)
    r,g,b=x01[:,0:1],x01[:,1:2],x01[:,2:3]
    return 0.299*r+0.587*g+0.114*b
def attr_saturation(x):   # "color" attribute: mean HSV saturation proxy in [0,1]
    x=img01(x); mx=x.max(1,keepdim=True).values; mn=x.min(1,keepdim=True).values
    return ((mx-mn)/(mx+1e-6)).mean(dim=[1,2,3])
def attr_complexity(x):   # normalized total variation (edge richness) in [0,1]
    l=luminance(img01(x))
    dh=(l[:,:,:,1:]-l[:,:,:,:-1]).abs().mean(dim=[1,2,3])
    dv=(l[:,:,1:,:]-l[:,:,:-1,:]).abs().mean(dim=[1,2,3])
    return (2.0*(dh+dv)).clamp(0,1)
def attr_density(x):      # ink/foreground fraction proxy: darkness coverage in [0,1]
    l=luminance(img01(x))
    return (1.0-l).mean(dim=[1,2,3]).clamp(0,1)
def attr_symmetry(x):     # mean of horizontal+vertical mirror agreement in [0,1]
    l=luminance(img01(x))
    sh=1.0-(l-torch.flip(l,dims=[3])).abs().mean(dim=[1,2,3])
    sv=1.0-(l-torch.flip(l,dims=[2])).abs().mean(dim=[1,2,3])
    return (0.5*(sh+sv)).clamp(0,1)
def attr_contrast(x):     # std of luminance in [0,1] (used inside aesthetic score)
    return (2.0*luminance(img01(x)).std(dim=[1,2,3])).clamp(0,1)
def continuous_attrs(x):  # (B,4): color, complexity, density, symmetry
    return torch.stack([attr_saturation(x),attr_complexity(x),attr_density(x),attr_symmetry(x)],dim=1)
print("Aesthetic-attribute functions defined (color/complexity/density/symmetry/contrast).")

Aesthetic-attribute functions defined (color/complexity/density/symmetry/contrast).


## 13 — Dataset class + condition tensors

In [19]:

_tf = transforms.Compose([
    transforms.Resize((CFG["image_size"],CFG["image_size"]),
                      interpolation=transforms.InterpolationMode.BICUBIC, antialias=True),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3),   # -> [-1,1]
])
class BatikDataset(Dataset):
    def __init__(self, df):
        self.paths=df["path"].tolist(); self.motif=df["motif_idx"].tolist()
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        x=_tf(Image.open(self.paths[i]).convert("RGB"))
        return x, torch.tensor(self.motif[i],dtype=torch.long)
def build_condition(x_imgs, motif_idx):
    """Full condition tensor c = [ one_hot(style) | color,complexity,density,symmetry ] -> (B, NUM_MOTIFS+4)."""
    oh=F.one_hot(motif_idx.to(torch.long), num_classes=NUM_MOTIFS).float()
    attrs=continuous_attrs(x_imgs).to(oh.device)
    return torch.cat([oh, attrs], dim=1)
COND_DIM = NUM_MOTIFS + 4
print("Dataset + conditioning ready. COND_DIM =", COND_DIM)

Dataset + conditioning ready. COND_DIM = 9


## 14 — DataLoaders

In [20]:

def make_loader(df, shuffle, bs=None):
    return DataLoader(BatikDataset(df), batch_size=bs or CFG["batch_size"], shuffle=shuffle,
                      num_workers=CFG["num_workers"], pin_memory=torch.cuda.is_available(),
                      drop_last=shuffle, worker_init_fn=seed_worker, generator=G_SEED,
                      persistent_workers=CFG["num_workers"]>0)
train_loader=make_loader(train_df, True)
val_loader  =make_loader(val_df, False)
test_loader =make_loader(test_df, False)
print("Batches -> train:%d val:%d test:%d"%(len(train_loader),len(val_loader),len(test_loader)))
# fixed deterministic condition pools (sampled from REAL data) for generation during eval
def condition_pool(df, n):
    idx=np.random.default_rng(CFG["seed"]).choice(len(df), size=min(n,len(df)), replace=False)
    ld=DataLoader(BatikDataset(df.iloc[idx].reset_index(drop=True)), batch_size=64, shuffle=False)
    cs=[]
    for xb,mb in ld: cs.append(build_condition(xb,mb))
    return torch.cat(cs,0)
print("condition_pool() ready.")

Batches -> train:62 val:10 test:20
condition_pool() ready.


## 15 — Feature-extraction utilities (shared Inception features, cached)

In [21]:

# One shared InceptionV3 pool3 (2048-d) extractor for diversity / novelty / MMD (fixed across ALL experiments).
class InceptionFeatures(nn.Module):
    def __init__(self):
        super().__init__()
        net=torchvision.models.inception_v3(weights=torchvision.models.Inception_V3_Weights.IMAGENET1K_V1,
                                             aux_logits=True)
        net.fc=nn.Identity(); net.eval()
        self.net=net
        for p in self.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def forward(self, x01):  # x01 in [0,1], (B,3,H,W)
        x=F.interpolate(x01, size=(299,299), mode="bilinear", align_corners=False)
        x=(x-0.5)/0.5
        return self.net(x)
_INCEPTION=None
def inception():
    global _INCEPTION
    if _INCEPTION is None: _INCEPTION=InceptionFeatures().to(DEVICE)
    return _INCEPTION
@torch.no_grad()
def extract_features(images01, bs=64):
    net=inception(); out=[]
    for i in range(0,images01.shape[0],bs):
        out.append(net(images01[i:i+bs].to(DEVICE)).cpu())
    return torch.cat(out,0)
@torch.no_grad()
def features_from_loader(loader, max_n=None):
    net=inception(); out=[]; n=0
    for xb,_ in loader:
        out.append(net(img01(xb).to(DEVICE)).cpu()); n+=xb.shape[0]
        if max_n and n>=max_n: break
    f=torch.cat(out,0)
    return f[:max_n] if max_n else f
print("Inception feature extractor ready (lazy-loaded on first use).")

Inception feature extractor ready (lazy-loaded on first use).


## 16 — FID (Fréchet Inception Distance) — lower is better
Computed from the shared cached Inception features (so FID/KID/diversity/novelty/MMD all use the
SAME extractor). Real features are cached per split and never recomputed unnecessarily.

In [22]:

from scipy import linalg
def _mu_sigma(feats):
    f=feats.double().numpy(); mu=f.mean(0); sig=np.cov(f, rowvar=False)
    return mu, sig
def fid_from_feats(fake_feats, real_mu, real_sigma):
    mu2,sig2=_mu_sigma(fake_feats)
    diff=real_mu-mu2
    covmean,_=linalg.sqrtm(real_sigma.dot(sig2), disp=False)
    if np.iscomplexobj(covmean): covmean=covmean.real
    fid=float(diff.dot(diff)+np.trace(real_sigma+sig2-2.0*covmean))
    return max(fid,0.0)
print("FID defined (feature-based, cached real mu/sigma).")

FID defined (feature-based, cached real mu/sigma).


## 17 — KID (Kernel Inception Distance) — lower is better
Unbiased polynomial-kernel estimator over repeated random subsets; mean ± std computed from ACTUAL
resampling (no fabricated CIs). Same features/protocol for every experiment.

In [23]:

def _poly_kernel(x,y):
    d=x.shape[1]; return (x@y.T/d + 1.0)**3
def kid_from_feats(fake_feats, real_feats, subset=None, n_subsets=100, seed=CFG["seed"]):
    subset=subset or CFG["kid_subset"]
    x=real_feats.double().numpy(); y=fake_feats.double().numpy()
    n=min(subset, x.shape[0], y.shape[0]); rng=np.random.default_rng(seed); vals=[]
    if n<2: return float("nan"), float("nan")
    for _ in range(n_subsets):
        xi=x[rng.choice(x.shape[0],n,replace=False)]; yi=y[rng.choice(y.shape[0],n,replace=False)]
        kxx=_poly_kernel(xi,xi); kyy=_poly_kernel(yi,yi); kxy=_poly_kernel(xi,yi)
        m=n
        t=(kxx.sum()-np.trace(kxx))/(m*(m-1)) + (kyy.sum()-np.trace(kyy))/(m*(m-1)) - 2*kxy.mean()
        vals.append(t)
    return float(np.mean(vals)), float(np.std(vals))
print("KID defined (poly kernel, real resampling for mean/std).")

KID defined (poly kernel, real resampling for mean/std).


## 18 — LPIPS — two DISTINCT measurements (directionality documented)
* `LPIPS_real_similarity` — perceptual distance of generated to real (LOWER = closer to real).
* `LPIPS_generated_diversity` — perceptual distance between generated pairs (HIGHER = more diverse).
These are different quantities and are reported separately.

In [24]:

import lpips as _lpips_pkg
_LPIPS=None
def lpips_net():
    global _LPIPS
    if _LPIPS is None: _LPIPS=_lpips_pkg.LPIPS(net="alex").to(DEVICE).eval()
    return _LPIPS
@torch.no_grad()
def lpips_real_similarity(fake_m11, real_m11, n_pairs=256):
    net=lpips_net(); n=min(n_pairs, fake_m11.shape[0], real_m11.shape[0])
    rng=np.random.default_rng(CFG["seed"])
    fi=rng.choice(fake_m11.shape[0],n,replace=False); ri=rng.choice(real_m11.shape[0],n,replace=False)
    d=[]
    for i in range(0,n,64):
        a=fake_m11[fi[i:i+64]].to(DEVICE); b=real_m11[ri[i:i+64]].to(DEVICE)
        d.append(net(a,b).flatten().cpu())
    return float(torch.cat(d).mean())
@torch.no_grad()
def lpips_generated_diversity(fake_m11, n_pairs=256):
    net=lpips_net(); N=fake_m11.shape[0]
    if N<2: return float("nan")
    rng=np.random.default_rng(CFG["seed"]+1)
    a_idx=rng.integers(0,N,n_pairs); b_idx=rng.integers(0,N,n_pairs)
    keep=a_idx!=b_idx; a_idx,b_idx=a_idx[keep],b_idx[keep]; d=[]
    for i in range(0,len(a_idx),64):
        a=fake_m11[a_idx[i:i+64]].to(DEVICE); b=fake_m11[b_idx[i:i+64]].to(DEVICE)
        d.append(net(a,b).flatten().cpu())
    return float(torch.cat(d).mean()) if d else float("nan")
print("LPIPS defined (real_similarity=lower better; generated_diversity=higher better).")

LPIPS defined (real_similarity=lower better; generated_diversity=higher better).


## 19 — Diversity (feature-space pairwise distance) — higher better *within quality constraints*

In [25]:

def diversity_from_feats(feats, max_n=512, seed=CFG["seed"]):
    n=feats.shape[0]
    if n<2: return dict(diversity_mean=float("nan"), diversity_median=float("nan"))
    if n>max_n:
        idx=np.random.default_rng(seed).choice(n,max_n,replace=False); feats=feats[idx]
    with torch.no_grad():
        d=torch.cdist(feats.float(), feats.float(), p=2)
        iu=torch.triu_indices(d.shape[0], d.shape[0], offset=1)
        vals=d[iu[0],iu[1]]
    return dict(diversity_mean=float(vals.mean()), diversity_median=float(vals.median()))
print("Diversity defined.")

Diversity defined.


## 20 — Novelty vs TRAINING data only — memorization guard
`novelty(G) = 1 - max cosine_similarity(emb(G), emb(train))`. Batched NN search (memory-safe).
High similarity => possible memorization; interpret jointly with realism/adherence.

In [26]:

def novelty_vs_train(fake_feats, train_feats, chunk=2048):
    fn=F.normalize(fake_feats.float(),dim=1); tn=F.normalize(train_feats.float(),dim=1)
    max_sim=torch.full((fn.shape[0],), -1.0)
    for i in range(0, tn.shape[0], chunk):
        sim=fn @ tn[i:i+chunk].T           # (F, chunk)
        max_sim=torch.maximum(max_sim, sim.max(dim=1).values)
    nov=1.0-max_sim
    return dict(novelty_mean=float(nov.mean()), novelty_median=float(nov.median()),
                novelty_min=float(nov.min()), nn_similarity_mean=float(max_sim.mean()))
print("Novelty defined (training-reference only, batched).")

Novelty defined (training-reference only, batched).


## 21 — Computational Aesthetic Score — [0,1], higher better
Composite of objective descriptors (contrast, complexity, symmetry, colorfulness). This is a
**Computational** score (no human ratings, no pretrained aesthetic model claimed).

In [27]:

@torch.no_grad()
def aesthetic_score(imgs):   # imgs in [-1,1] or [0,1]
    x=img01(imgs)
    comps=torch.stack([attr_contrast(x), attr_complexity(x), attr_symmetry(x), attr_saturation(x)],dim=1)
    score=comps.mean(dim=1).clamp(0,1)        # equal-weight composite
    return float(score.mean())
print("Computational Aesthetic Score defined.")

Computational Aesthetic Score defined.


## 22 — Seamless Score — [0,1], higher better (independently computed, NOT the train loss)

In [28]:

@torch.no_grad()
def seamless_score(imgs):
    x=img01(imgs)
    h_err=(x[:,:,:,0]-x[:,:,:,-1]).abs().mean(dim=[1,2])   # left vs right column
    v_err=(x[:,:,0,:]-x[:,:,-1,:]).abs().mean(dim=[1,2])   # top vs bottom row
    score=(1.0-0.5*(h_err+v_err)).clamp(0,1)
    return dict(seamless_score=float(score.mean()),
                horizontal_seam_error=float(h_err.mean()),
                vertical_seam_error=float(v_err.mean()))
print("Seamless Score defined.")

Seamless Score defined.


## 23 — Condition Adherence — [0,1], higher better
Continuous attributes (color/complexity/density/symmetry) are re-measured from generated images and
compared to the requested condition. Style adherence = nearest real-class centroid (Inception space)
matching the requested motif. If an attribute can't be reliably measured it is reported as
`unavailable`.

In [29]:

_CLASS_CENTROIDS=None
def build_class_centroids(train_loader_):
    global _CLASS_CENTROIDS
    net=inception(); sums={m:torch.zeros(2048) for m in range(NUM_MOTIFS)}; cnt=Counter()
    with torch.no_grad():
        for xb,mb in train_loader_:
            f=net(img01(xb).to(DEVICE)).cpu()
            for k in range(xb.shape[0]):
                sums[int(mb[k])]+=f[k]; cnt[int(mb[k])]+=1
    _CLASS_CENTROIDS=torch.stack([F.normalize(sums[m]/max(cnt[m],1),dim=0) for m in range(NUM_MOTIFS)])
    return _CLASS_CENTROIDS
@torch.no_grad()
def condition_adherence(fake_imgs, requested_cond, fake_feats=None):
    # continuous part = last 4 columns of the condition
    req_attrs=requested_cond[:, NUM_MOTIFS:NUM_MOTIFS+4].to(fake_imgs.device)
    meas=continuous_attrs(fake_imgs).to(fake_imgs.device)
    n=min(req_attrs.shape[0], meas.shape[0])
    cont=float((1.0-(meas[:n]-req_attrs[:n]).abs()).clamp(0,1).mean())
    style=float("nan")
    if _CLASS_CENTROIDS is not None and fake_feats is not None:
        req_style=requested_cond[:n,:NUM_MOTIFS].argmax(1)
        fn=F.normalize(fake_feats[:n].float(),dim=1)
        pred=(fn @ _CLASS_CENTROIDS.T).argmax(1)
        style=float((pred.cpu()==req_style.cpu()).float().mean())
    combined = cont if math.isnan(style) else 0.5*(cont+style)
    return dict(condition_adherence=combined, adherence_continuous=cont,
                adherence_style=("unavailable" if math.isnan(style) else style))
print("Condition Adherence defined.")

Condition Adherence defined.


## 24 — MMD (RBF kernel) — lower is better
Explicit NumPy→Torch conversion (never pass numpy to torch.cdist). Bandwidth = median-distance
heuristic (recorded). Memory-safe subsampling.

In [30]:

def _as_tensor(a):
    if isinstance(a, np.ndarray): a=torch.from_numpy(a)
    return a.float()
def mmd_rbf(fake_feats, real_feats, max_n=512, seed=CFG["seed"]):
    x=_as_tensor(fake_feats); y=_as_tensor(real_feats)
    rng=np.random.default_rng(seed)
    if x.shape[0]>max_n: x=x[rng.choice(x.shape[0],max_n,replace=False)]
    if y.shape[0]>max_n: y=y[rng.choice(y.shape[0],max_n,replace=False)]
    with torch.no_grad():
        dxx=torch.cdist(x,x)**2; dyy=torch.cdist(y,y)**2; dxy=torch.cdist(x,y)**2
        med=torch.median(torch.cat([dxx.flatten(), dyy.flatten(), dxy.flatten()]))
        sigma2=float(med)+1e-8               # median heuristic (squared-distance)
        kxx=torch.exp(-dxx/(2*sigma2)); kyy=torch.exp(-dyy/(2*sigma2)); kxy=torch.exp(-dxy/(2*sigma2))
        m,n=x.shape[0],y.shape[0]
        mmd=(kxx.sum()-kxx.diag().sum())/(m*(m-1)) + (kyy.sum()-kyy.diag().sum())/(n*(n-1)) - 2*kxy.mean()
    return dict(mmd=float(max(mmd,0.0)), mmd_bandwidth_sigma2=sigma2)
print("MMD defined (median-heuristic bandwidth, torch-safe).")

MMD defined (median-heuristic bandwidth, torch-safe).


## 25 — Metric sanity tests (must pass before training)

In [33]:
def metric_sanity_tests():
    print("Running metric sanity checks ..."); ok=True
    xb,mb=next(iter(val_loader)); xb=xb[:32]
    f=extract_features(img01(xb))
    mu,sig=_mu_sigma(f); fid_rr=fid_from_feats(f, mu, sig)
    print("  FID(real,real) ~0 :", round(fid_rr,4)); ok &= fid_rr<5.0
    mmd_ii=mmd_rbf(f,f)["mmd"]; print("  MMD(identical) ~0 :", round(mmd_ii,6)); ok &= mmd_ii<1e-3
    kid_rr,_=kid_from_feats(f,f,subset=16,n_subsets=20); print("  KID(real,real) ~0 :", round(kid_rr,5)); ok &= abs(kid_rr)<0.05
    # LPIPS on TRULY identical vs noisy (aligned pairs, using the net directly)
    _net=lpips_net()
    with torch.no_grad():
        _a=xb[:16].to(DEVICE)
        lp_id=float(_net(_a,_a).mean())
        _noisy=(_a+0.5*torch.randn_like(_a)).clamp(-1,1)
        lp_noisy=float(_net(_a,_noisy).mean())
    print("  LPIPS(identical) ~0 :", round(lp_id,4), "| LPIPS(noisy) >:", round(lp_noisy,4))
    ok &= (lp_id<0.05) and (lp_noisy>lp_id)
    per=torch.sin(torch.linspace(0,2*math.pi,CFG["image_size"])).view(1,1,1,-1).repeat(1,3,CFG["image_size"],1)
    ss=seamless_score(per)["seamless_score"]; print("  Seamless(periodic) high :", round(ss,4)); ok &= ss>0.9
    a=aesthetic_score(xb); print("  Aesthetic in [0,1] :", round(a,4)); ok &= (0.0<=a<=1.0)
    print("SANITY:", "PASS" if ok else "FAIL")
    assert ok, "Metric sanity tests failed — do not proceed."
    return ok

metric_sanity_tests()


Running metric sanity checks ...
  FID(real,real) ~0 : 0.0
  MMD(identical) ~0 : 0.0
  KID(real,real) ~0 : -0.0144
  LPIPS(identical) ~0 : 0.0 | LPIPS(noisy) >: 0.4294
  Seamless(periodic) high : 1.0
  Aesthetic in [0,1] : 0.479
SANITY: PASS


True

## 26 — Latent utilities (disentangled sub-spaces)

In [34]:

LATENT_SLICES=OrderedDict(
    structure=(0, CFG["z_structure"]),
    style=(CFG["z_structure"], CFG["z_structure"]+CFG["z_style"]),
    color=(CFG["z_structure"]+CFG["z_style"], CFG["z_structure"]+CFG["z_style"]+CFG["z_color"]),
    texture=(CFG["z_structure"]+CFG["z_style"]+CFG["z_color"], CFG["latent_dim"]),
)
def sample_z(bs): return torch.randn(bs, CFG["latent_dim"], device=DEVICE)
def sample_cond(bs, cond_pool):
    idx=torch.randint(0, cond_pool.shape[0], (bs,))
    return cond_pool[idx].to(DEVICE)
print("Latent slices:", dict(LATENT_SLICES))

Latent slices: {'structure': (0, 48), 'style': (48, 80), 'color': (80, 104), 'texture': (104, 128)}


## 27 — Generator & Discriminator (flag-driven; baseline = all flags off)
One flag-driven architecture keeps ablations fair (identical capacity/preprocessing). Baseline uses
`use_cond=False`; ACDA variants set `use_cond=True`. Hinge GAN. Fits L4 easily at 64×64.

In [35]:

def dconv(i,o): return nn.Sequential(nn.Conv2d(i,o,4,2,1), nn.LeakyReLU(0.2,True))
class Generator(nn.Module):
    def __init__(self, use_cond=False, base=64, cond_embed=128):
        super().__init__()
        self.use_cond=use_cond
        if use_cond: self.cond_emb=nn.Sequential(nn.Linear(COND_DIM, cond_embed), nn.ReLU(True))
        in_dim = CFG["latent_dim"] + (cond_embed if use_cond else 0)
        self.fc=nn.Linear(in_dim, base*8*4*4)
        self.net=nn.Sequential(
            nn.BatchNorm2d(base*8), nn.ReLU(True),
            nn.ConvTranspose2d(base*8, base*4,4,2,1), nn.BatchNorm2d(base*4), nn.ReLU(True),   #8
            nn.ConvTranspose2d(base*4, base*2,4,2,1), nn.BatchNorm2d(base*2), nn.ReLU(True),   #16
            nn.ConvTranspose2d(base*2, base,  4,2,1), nn.BatchNorm2d(base),   nn.ReLU(True),   #32
            nn.ConvTranspose2d(base,   CFG["channels"],4,2,1), nn.Tanh())                      #64
        self.base=base
    def forward(self, z, cond=None):
        h = z if not self.use_cond else torch.cat([z, self.cond_emb(cond)], dim=1)
        h = self.fc(h).view(-1, self.base*8, 4, 4)
        return self.net(h)
class Discriminator(nn.Module):
    def __init__(self, use_cond=False, base=64):
        super().__init__()
        self.use_cond=use_cond
        self.body=nn.Sequential(
            nn.Conv2d(CFG["channels"], base,4,2,1), nn.LeakyReLU(0.2,True),   #32
            dconv(base, base*2), dconv(base*2, base*4), dconv(base*4, base*8))#16,8,4
        self.feat_dim=base*8*4*4
        self.fc=nn.Linear(self.feat_dim, 1)
        if use_cond: self.proj=nn.Linear(COND_DIM, self.feat_dim)
    def forward(self, x, cond=None):
        h=self.body(x).flatten(1)
        out=self.fc(h).squeeze(1)
        if self.use_cond and cond is not None:
            out=out + (self.proj(cond)*h).sum(1)
        return out, h
print("Generator/Discriminator defined.")

Generator/Discriminator defined.


## 28 — Disentanglement encoder Q + model factory

In [36]:

class LatentEncoder(nn.Module):   # InfoGAN-style Q: predicts z back from image (disentanglement)
    def __init__(self, base=64):
        super().__init__()
        self.body=nn.Sequential(
            nn.Conv2d(CFG["channels"], base,4,2,1), nn.LeakyReLU(0.2,True),
            dconv(base,base*2), dconv(base*2,base*4), dconv(base*4,base*8))
        self.head=nn.Linear(base*8*4*4, CFG["latent_dim"])
    def forward(self,x): return self.head(self.body(x).flatten(1))

FLAG_TABLE=OrderedDict(
    A_baseline          =dict(cond=False,disent=False,seam=False,aest=False,nov=False),
    B_conditioning      =dict(cond=True, disent=False,seam=False,aest=False,nov=False),
    C_disentangled      =dict(cond=True, disent=True, seam=False,aest=False,nov=False),
    D_seamless          =dict(cond=True, disent=True, seam=True, aest=False,nov=False),
    E_aesthetic         =dict(cond=True, disent=True, seam=True, aest=True, nov=False),
    F_full_ACDA         =dict(cond=True, disent=True, seam=True, aest=True, nov=True),
)
def build_models(flags):
    G=Generator(use_cond=flags["cond"]).to(DEVICE)
    D=Discriminator(use_cond=flags["cond"]).to(DEVICE)
    Q=LatentEncoder().to(DEVICE) if flags["disent"] else None
    return G,D,Q
def count_params(m): return sum(p.numel() for p in m.parameters()) if m is not None else 0
print("Ablation flag table:"); [print("  ", k, v) for k,v in FLAG_TABLE.items()]

Ablation flag table:
   A_baseline {'cond': False, 'disent': False, 'seam': False, 'aest': False, 'nov': False}
   B_conditioning {'cond': True, 'disent': False, 'seam': False, 'aest': False, 'nov': False}
   C_disentangled {'cond': True, 'disent': True, 'seam': False, 'aest': False, 'nov': False}
   D_seamless {'cond': True, 'disent': True, 'seam': True, 'aest': False, 'nov': False}
   E_aesthetic {'cond': True, 'disent': True, 'seam': True, 'aest': True, 'nov': False}
   F_full_ACDA {'cond': True, 'disent': True, 'seam': True, 'aest': True, 'nov': True}


[None, None, None, None, None, None]

## 29 — Loss definitions
`L_GAN` hinge; `L_SEAM`=periodic boundary L1; `L_AEST`=1−aesthetic proxy; `L_NOV`=batch feature
repulsion (mean pairwise cosine similarity of D features, minimized); `L_DIS`=MSE(Q(fake), z).

In [37]:

def d_hinge(real_logit, fake_logit):
    return F.relu(1.0-real_logit).mean() + F.relu(1.0+fake_logit).mean()
def g_adv(fake_logit): return -fake_logit.mean()
def aesthetic_proxy(x):                       # differentiable, [0,1]
    xx=img01(x)
    return torch.stack([attr_contrast(xx),attr_complexity(xx),attr_symmetry(xx),attr_saturation(xx)],1).mean(1)
def loss_aest(fake): return (1.0-aesthetic_proxy(fake)).mean()
def loss_seam(fake):
    x=img01(fake)
    return (x[:,:,:,0]-x[:,:,:,-1]).abs().mean() + (x[:,:,0,:]-x[:,:,-1,:]).abs().mean()
def loss_nov(h_feats):                        # repulsion: minimize mean pairwise cosine similarity
    hn=F.normalize(h_feats,dim=1); sim=hn@hn.T
    n=hn.shape[0]; off=(sim.sum()-sim.diag().sum())/(n*(n-1)+1e-8)
    return off
def loss_dis(Q, fake, z): return F.mse_loss(Q(fake), z)
print("Losses defined.")

Losses defined.


## 30 — Optimizers

In [38]:

def build_optimizers(G,D,Q):
    optG=torch.optim.Adam(G.parameters(), lr=CFG["lr_g"], betas=(CFG["beta1"],CFG["beta2"]))
    d_params=list(D.parameters()) + (list(Q.parameters()) if Q is not None else [])
    optD=torch.optim.Adam(d_params, lr=CFG["lr_d"], betas=(CFG["beta1"],CFG["beta2"]))
    return optG, optD
print("Optimizer factory ready.")

Optimizer factory ready.


## 31 — Schedulers (ReduceLROnPlateau on val_FID)

In [39]:

def build_schedulers(optG, optD):
    mk=lambda opt: torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=CFG["lr_factor"], patience=CFG["lr_patience"], min_lr=CFG["min_lr"])
    return mk(optG), mk(optD)
print("Scheduler factory ready.")

Scheduler factory ready.


## 32 — Checkpointing + resume (full state)

In [40]:

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(),
                torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)
def set_rng_state(s):
    try:
        random.setstate(s["python"]); np.random.set_state(s["numpy"])
        torch.set_rng_state(s["torch"].cpu() if hasattr(s["torch"],"cpu") else s["torch"])
        if s.get("cuda") is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(s["cuda"])
    except Exception as e: print("RNG restore warning:", e)
def save_ckpt(path, ep, bundle, history, best_metric, best_epoch, es_counter):
    G,D,Q,optG,optD,schG,schD,scaler,flags=bundle
    torch.save(dict(epoch=ep, generator_state_dict=G.state_dict(),
        discriminator_state_dict=D.state_dict(),
        q_state_dict=(Q.state_dict() if Q is not None else None),
        optimizer_G_state_dict=optG.state_dict(), optimizer_D_state_dict=optD.state_dict(),
        scheduler_G_state_dict=schG.state_dict(), scheduler_D_state_dict=schD.state_dict(),
        scaler_state_dict=scaler.state_dict(), training_history=history,
        best_metric=best_metric, best_epoch=best_epoch, early_stopping_counter=es_counter,
        rng=rng_state(), config=CFG, flags=flags, latent_slices=dict(LATENT_SLICES),
        motifs=MOTIFS), path+".tmp"); os.replace(path+".tmp", path)
def load_ckpt(path, bundle, map_location=DEVICE):
    G,D,Q,optG,optD,schG,schD,scaler,flags=bundle
    st=torch.load(path, map_location=map_location, weights_only=False)
    G.load_state_dict(st["generator_state_dict"]); D.load_state_dict(st["discriminator_state_dict"])
    if Q is not None and st.get("q_state_dict"): Q.load_state_dict(st["q_state_dict"])
    optG.load_state_dict(st["optimizer_G_state_dict"]); optD.load_state_dict(st["optimizer_D_state_dict"])
    schG.load_state_dict(st["scheduler_G_state_dict"]); schD.load_state_dict(st["scheduler_D_state_dict"])
    scaler.load_state_dict(st["scaler_state_dict"]); set_rng_state(st.get("rng"))
    return st
print("Checkpoint system ready.")

Checkpoint system ready.


## 33 — Training utilities (one step / one epoch, AMP, hinge, per-ablation losses)

In [41]:

def generate(G, flags, n, cond_pool):
    z=sample_z(n); c=sample_cond(n, cond_pool) if flags["cond"] else None
    return G(z, c), z, c
def train_one_epoch(bundle, loader, cond_pool):
    G,D,Q,optG,optD,schG,schD,scaler,flags=bundle
    G.train(); D.train();  Q and Q.train()
    agg=Counter(); nb=0
    for xb,mb in loader:
        xb=xb.to(DEVICE, non_blocking=True)
        cond = build_condition(xb, mb.to(DEVICE)) if flags["cond"] else None
        bs=xb.size(0)
        # ---------- D ----------
        optD.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=CFG["use_amp"] and torch.cuda.is_available()):
            z=sample_z(bs); fake=G(z, cond).detach()
            r_logit,_=D(xb, cond); f_logit,_=D(fake, cond)
            d_loss=d_hinge(r_logit, f_logit)
            if flags["disent"] and Q is not None:
                d_loss=d_loss + CFG["lambda_dis"]*loss_dis(Q, fake, z)
        scaler.scale(d_loss).backward(); scaler.step(optD); scaler.update()
        # ---------- G ----------
        optG.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=CFG["use_amp"] and torch.cuda.is_available()):
            z=sample_z(bs); fake=G(z, cond)
            f_logit,h=D(fake, cond)
            gan=g_adv(f_logit); gl=gan
            la=torch.tensor(0.,device=DEVICE); ld=torch.tensor(0.,device=DEVICE)
            ls=torch.tensor(0.,device=DEVICE); ln=torch.tensor(0.,device=DEVICE)
            if flags["seam"]: ls=loss_seam(fake); gl=gl+CFG["lambda_seam"]*ls
            if flags["aest"]: la=loss_aest(fake); gl=gl+CFG["lambda_aest"]*la
            if flags["nov"]:  ln=loss_nov(h);     gl=gl+CFG["lambda_nov"]*ln
            if flags["disent"] and Q is not None: ld=loss_dis(Q, fake, z); gl=gl+CFG["lambda_dis"]*ld
        scaler.scale(gl).backward(); scaler.step(optG); scaler.update()
        agg["generator_loss"]+=float(gl); agg["discriminator_loss"]+=float(d_loss)
        agg["gan_loss"]+=float(gan); agg["aesthetic_loss"]+=float(la); agg["disentanglement_loss"]+=float(ld)
        agg["seam_loss"]+=float(ls); agg["novelty_loss"]+=float(ln); nb+=1
        if not torch.isfinite(gl) or not torch.isfinite(d_loss):
            raise FloatingPointError("NaN/Inf loss encountered — aborting (emergency save handled by caller).")
    return {k: v/max(nb,1) for k,v in agg.items()}
print("Training utilities ready.")

Training utilities ready.


## 34 — Validation-loss pass + generative-metric evaluation (fixed subsets, cached real features)

In [42]:

@torch.no_grad()
def val_losses(bundle, loader, cond_pool):
    G,D,Q,*_=bundle; flags=bundle[8]; G.eval(); D.eval();  Q and Q.eval()
    agg=Counter(); nb=0
    for xb,mb in loader:
        xb=xb.to(DEVICE); cond=build_condition(xb, mb.to(DEVICE)) if flags["cond"] else None
        bs=xb.size(0); z=sample_z(bs); fake=G(z,cond)
        r_logit,_=D(xb,cond); f_logit,h=D(fake,cond)
        d_loss=d_hinge(r_logit,f_logit); gan=g_adv(f_logit); gl=gan
        la=loss_aest(fake) if flags["aest"] else torch.tensor(0.,device=DEVICE)
        ls=loss_seam(fake) if flags["seam"] else torch.tensor(0.,device=DEVICE)
        ln=loss_nov(h)     if flags["nov"]  else torch.tensor(0.,device=DEVICE)
        ld=loss_dis(Q,fake,z) if (flags["disent"] and Q is not None) else torch.tensor(0.,device=DEVICE)
        gl=gan+CFG["lambda_seam"]*ls+CFG["lambda_aest"]*la+CFG["lambda_nov"]*ln+CFG["lambda_dis"]*ld
        agg["val_generator_loss"]+=float(gl); agg["val_discriminator_loss"]+=float(d_loss)
        agg["val_gan_loss"]+=float(gan); agg["val_aesthetic_loss"]+=float(la)
        agg["val_disentanglement_loss"]+=float(ld); agg["val_seam_loss"]+=float(ls)
        agg["val_novelty_loss"]+=float(ln); nb+=1
    return {k:v/max(nb,1) for k,v in agg.items()}

# ---- dataset-level metric cache (built ONCE, shared by ALL ablations = fair) ----
METRIC_CACHE={}
def build_metric_cache():
    def subset_loader(df, n):
        idx=np.random.default_rng(CFG["seed"]).choice(len(df), size=min(n,len(df)), replace=False)
        return DataLoader(BatikDataset(df.iloc[idx].reset_index(drop=True)), batch_size=64, shuffle=False)
    tl=subset_loader(train_df, CFG["eval_subset"]); vl=subset_loader(val_df, CFG["eval_subset"])
    tr_feats=features_from_loader(tl); va_feats=features_from_loader(vl)
    # real images ([-1,1]) for LPIPS
    def imgs_from(loader_):
        xs=[xb for xb,_ in loader_]; return torch.cat(xs,0)
    METRIC_CACHE.update(
        train_feats=tr_feats, val_feats=va_feats,
        train_mu_sigma=_mu_sigma(tr_feats), val_mu_sigma=_mu_sigma(va_feats),
        train_ref_feats=features_from_loader(make_loader(train_df, False), max_n=2000),  # novelty reference (TRAIN only)
        val_real_imgs=imgs_from(vl), train_real_imgs=imgs_from(tl),
        train_cond=condition_pool(train_df, CFG["eval_subset"]),
        val_cond=condition_pool(val_df, CFG["eval_subset"]))
    build_class_centroids(make_loader(train_df, False))
    print("Metric cache built (train/val features, novelty ref, class centroids).")

@torch.no_grad()
def eval_metrics(bundle, split="val", n_gen=None, return_images=False):
    G=bundle[0]; flags=bundle[8]; G.eval()
    n_gen=n_gen or CFG["eval_n_gen"]
    cond_pool=METRIC_CACHE[split+"_cond"]
    real_ms=METRIC_CACHE[split+"_mu_sigma"]
    real_feats=METRIC_CACHE[split+"_feats"]
    real_imgs=METRIC_CACHE[split+"_real_imgs"]
    fakes=[]; conds=[]
    for i in range(0, n_gen, 64):
        b=min(64, n_gen-i); img,_,c=generate(G, flags, b, cond_pool)
        fakes.append(img.cpu()); conds.append((c.cpu() if c is not None else torch.zeros(b,COND_DIM)))
    fake=torch.cat(fakes,0); cond_used=torch.cat(conds,0)
    fake01=img01(fake); fake_feats=extract_features(fake01)
    fid=fid_from_feats(fake_feats, real_ms[0], real_ms[1])
    kid_m,kid_s=kid_from_feats(fake_feats, real_feats)
    div=diversity_from_feats(fake_feats); nov=novelty_vs_train(fake_feats, METRIC_CACHE["train_ref_feats"])
    mmd=mmd_rbf(fake_feats, real_feats)
    lp_sim=lpips_real_similarity(fake, real_imgs); lp_div=lpips_generated_diversity(fake)
    aes=aesthetic_score(fake); seam=seamless_score(fake)
    adh=condition_adherence(fake.to(DEVICE), cond_used.to(DEVICE), fake_feats=fake_feats) if flags["cond"] \
        else dict(condition_adherence=float("nan"), adherence_continuous=float("nan"), adherence_style="unavailable")
    out=dict(FID=fid, KID=kid_m, KID_std=kid_s, LPIPS=lp_sim, LPIPS_generated_diversity=lp_div,
             diversity=div["diversity_mean"], diversity_median=div["diversity_median"],
             novelty=nov["novelty_mean"], novelty_min=nov["novelty_min"], nn_similarity=nov["nn_similarity_mean"],
             aesthetic_score=aes, seamless_score=seam["seamless_score"],
             condition_adherence=adh["condition_adherence"], MMD=mmd["mmd"], mmd_sigma2=mmd["mmd_bandwidth_sigma2"])
    return (out, fake) if return_images else out
print("Validation + metric evaluation utilities ready.")

Validation + metric evaluation utilities ready.


## 35 — Plotting utilities (DPI=300, font=20)

In [43]:

plt.rcParams.update({"font.size":20,"axes.titlesize":20,"axes.labelsize":20,
                     "xtick.labelsize":20,"ytick.labelsize":20,"legend.fontsize":18,
                     "savefig.dpi":300,"figure.dpi":110})
def _line(ax, df, cols, labels, title, ylab):
    plotted=False
    for c,l in zip(cols,labels):
        if c in df.columns and df[c].notna().any():
            ax.plot(df["epoch"], df[c], marker="o", ms=3, lw=2, label=l); plotted=True
    ax.set_title(title); ax.set_xlabel("epoch"); ax.set_ylabel(ylab); ax.grid(alpha=.3)
    if plotted: ax.legend()
    return plotted
def plot_history(df, outdir, name):
    os.makedirs(outdir, exist_ok=True)
    specs=[("generator_loss",["train_generator_loss","val_generator_loss"],["train","val"],"Generator Loss"),
           ("discriminator_loss",["train_discriminator_loss","val_discriminator_loss"],["train","val"],"Discriminator Loss"),
           ("gan_loss",["train_gan_loss","val_gan_loss"],["train","val"],"GAN Loss"),
           ("aesthetic_loss",["train_aesthetic_loss","val_aesthetic_loss"],["train","val"],"Aesthetic Loss"),
           ("disentanglement_loss",["train_disentanglement_loss","val_disentanglement_loss"],["train","val"],"Disentanglement Loss"),
           ("seam_loss",["train_seam_loss","val_seam_loss"],["train","val"],"Seam Loss"),
           ("novelty_loss",["train_novelty_loss","val_novelty_loss"],["train","val"],"Novelty Loss"),
           ("FID",["train_FID","val_FID"],["train","val"],"FID"),
           ("KID",["train_KID","val_KID"],["train","val"],"KID"),
           ("LPIPS",["train_LPIPS","val_LPIPS"],["train","val"],"LPIPS (real-sim)"),
           ("diversity",["train_diversity","val_diversity"],["train","val"],"Diversity"),
           ("novelty",["train_novelty","val_novelty"],["train","val"],"Novelty"),
           ("aesthetic_score",["train_aesthetic_score","val_aesthetic_score"],["train","val"],"Aesthetic Score"),
           ("seamless_score",["train_seamless_score","val_seamless_score"],["train","val"],"Seamless Score"),
           ("condition_adherence",["train_condition_adherence","val_condition_adherence"],["train","val"],"Condition Adherence"),
           ("MMD",["train_MMD","val_MMD"],["train","val"],"MMD"),
           ("lr",["lr_G","lr_D"],["G","D"],"Learning Rate")]
    for fname,cols,labels,title in specs:
        fig,ax=plt.subplots(figsize=(9,6))
        if _line(ax, df, cols, labels, "%s: %s"%(name,title), title):
            fig.tight_layout(); fig.savefig(os.path.join(outdir,fname+".png"), dpi=300, bbox_inches="tight")
            try: fig.savefig(os.path.join(outdir,fname+".pdf"), bbox_inches="tight")
            except Exception: pass
        plt.close(fig)
def save_grid(imgs_m11, path, title="", nrow=8):
    x=img01(imgs_m11).cpu()
    grid=torchvision.utils.make_grid(x, nrow=nrow)
    fig=plt.figure(figsize=(10,10)); plt.imshow(grid.permute(1,2,0).numpy()); plt.axis("off")
    if title: plt.title(title)
    fig.tight_layout(); fig.savefig(path, dpi=300, bbox_inches="tight"); plt.close(fig)
print("Plotting utilities ready.")

Plotting utilities ready.


## 36 — Experiment runner + **Ablation A: Baseline GAN**
One runner trains any ablation (fair: same data/preprocessing/seed/eval). Selection by **val_FID**;
TEST is never touched here. Emergency checkpoint on NaN/Inf. Resume-capable.

In [44]:

NINE=["FID","KID","LPIPS","diversity","novelty","aesthetic_score","seamless_score","condition_adherence","MMD"]
RESUME=True
def composite_score(m):   # transparent, bounded [0,1]; for LOGGING (selection uses val_FID)
    w=CFG["composite_weights"]; import math as _m
    def hi(v): return 0.0 if (v is None or (isinstance(v,float) and _m.isnan(v))) else v
    fidn=1/(1+hi(m["FID"])/100.0); kidn=1/(1+max(hi(m["KID"]),0)*10); mmdn=1/(1+max(hi(m["MMD"]),0)*10)
    divn=1-_m.exp(-max(hi(m["diversity"]),0)/10.0)
    parts=dict(FID=fidn, KID=kidn, MMD=mmdn, diversity=divn,
               novelty=hi(m["novelty"]), aesthetic=hi(m["aesthetic_score"]),
               seamless=hi(m["seamless_score"]),
               condition_adherence=(0.0 if _m.isnan(hi(m["condition_adherence"])) else hi(m["condition_adherence"])))
    return float(sum(w[k]*parts[k] for k in w))
def print_epoch(name, ep, epochs, tr, trm, vl, vlm, optG, optD, best, best_ep, es, etime):
    L="="*80
    print(L); print("%s\nEPOCH %d / %d"%(name, ep, epochs)); print(L)
    print("\nTRAIN")
    print("Generator Loss:       %.4f"%tr["generator_loss"]);  print("Discriminator Loss:   %.4f"%tr["discriminator_loss"])
    print("GAN Loss:             %.4f"%tr["gan_loss"]);        print("Aesthetic Loss:       %.4f"%tr["aesthetic_loss"])
    print("Disentanglement Loss: %.4f"%tr["disentanglement_loss"]); print("Seam Loss:            %.4f"%tr["seam_loss"])
    print("Novelty Loss:         %.4f"%tr["novelty_loss"])
    for k in NINE: print("%-20s %.4f"%(k+":", trm[k]))
    print("\nVALIDATION")
    print("Generator Loss:       %.4f"%vl["val_generator_loss"]); print("Discriminator Loss:   %.4f"%vl["val_discriminator_loss"])
    print("GAN Loss:             %.4f"%vl["val_gan_loss"]);       print("Aesthetic Loss:       %.4f"%vl["val_aesthetic_loss"])
    print("Disentanglement Loss: %.4f"%vl["val_disentanglement_loss"]); print("Seam Loss:            %.4f"%vl["val_seam_loss"])
    print("Novelty Loss:         %.4f"%vl["val_novelty_loss"])
    for k in NINE: print("%-20s %.4f"%(k+":", vlm[k]))
    print("\nLR G: %.2e   LR D: %.2e"%(optG.param_groups[0]["lr"], optD.param_groups[0]["lr"]))
    print("Best Epoch: %d   Best %s: %.4f   Early Stop Counter: %d"%(best_ep,CFG["es_monitor"],best,es))
    print("Epoch Time: %.1fs"%etime); print(L+"\n")

def run_experiment(name, flags, epochs=None, resume=True):
    epochs=epochs or CFG["epochs"]
    ck=os.path.join(CHECKPOINT_ROOT, name); os.makedirs(ck, exist_ok=True)
    samp=os.path.join(OUTPUT_ROOT,"samples",name); os.makedirs(samp, exist_ok=True)
    hist_csv=os.path.join(OUTPUT_ROOT,"histories", name+"_training_history.csv")
    set_seed(CFG["seed"])                      # identical seed policy per ablation (fair)
    G,D,Q=build_models(flags); optG,optD=build_optimizers(G,D,Q); schG,schD=build_schedulers(optG,optD)
    scaler=torch.cuda.amp.GradScaler(enabled=CFG["use_amp"] and torch.cuda.is_available())
    bundle=[G,D,Q,optG,optD,schG,schD,scaler,flags]
    history=[]; start=0; best=float("inf"); best_ep=-1; es=0
    latest=os.path.join(ck,"latest_checkpoint.pt")
    if resume and os.path.isfile(latest):
        st=load_ckpt(latest, bundle); history=st.get("training_history",[]) or []
        start=int(st.get("epoch",0)); best=st.get("best_metric",float("inf"))
        best_ep=st.get("best_epoch",-1); es=int(st.get("early_stopping_counter",0))
        print("Resuming %s from epoch %d (best %s=%.4f)"%(name,start,CFG["es_monitor"],best))
    else:
        print("Starting %s from epoch 0"%name)
    for ep in range(start+1, epochs+1):
        t0=time.time()
        try:
            tr=train_one_epoch(bundle, train_loader, METRIC_CACHE["train_cond"])
        except FloatingPointError as e:
            save_ckpt(os.path.join(ck,"emergency_checkpoint.pt"), ep, bundle, history, best, best_ep, es)
            print("EMERGENCY (NaN/Inf):", e, "-> emergency_checkpoint.pt saved"); raise
        trm=eval_metrics(bundle, split="train"); vl=val_losses(bundle, val_loader, METRIC_CACHE["val_cond"])
        vlm=eval_metrics(bundle, split="val")
        if vlm["diversity"]<1e-3: print("  WARNING: near-zero diversity (possible mode collapse).")
        row=dict(epoch=ep,
            train_generator_loss=tr["generator_loss"], train_discriminator_loss=tr["discriminator_loss"],
            train_gan_loss=tr["gan_loss"], train_aesthetic_loss=tr["aesthetic_loss"],
            train_disentanglement_loss=tr["disentanglement_loss"], train_seam_loss=tr["seam_loss"],
            train_novelty_loss=tr["novelty_loss"],
            **{"train_"+k: trm[k] for k in NINE}, **vl, **{"val_"+k: vlm[k] for k in NINE},
            val_composite=composite_score(vlm),
            lr_G=optG.param_groups[0]["lr"], lr_D=optD.param_groups[0]["lr"], epoch_time=time.time()-t0)
        history.append(row); pd.DataFrame(history).to_csv(hist_csv, index=False)
        monitor=vlm["FID"]
        if monitor < best-1e-6:
            best=monitor; best_ep=ep; es=0
            save_ckpt(os.path.join(ck,"best_checkpoint.pt"), ep, bundle, history, best, best_ep, es)
        else: es+=1
        schG.step(monitor); schD.step(monitor)
        save_ckpt(latest, ep, bundle, history, best, best_ep, es)
        with torch.no_grad(): img,_,_=generate(G, flags, 64, METRIC_CACHE["val_cond"])
        save_grid(img, os.path.join(samp,"epoch_%04d.png"%ep), "%s ep%d"%(name,ep))
        print_epoch(name, ep, epochs, tr, trm, vl, vlm, optG, optD, best, best_ep, es, row["epoch_time"])
        if es>=CFG["es_patience"]:
            print("Early stopping %s at epoch %d."%(name,ep)); break
    tt=float(pd.DataFrame(history)["epoch_time"].sum()) if history else 0.0
    return dict(name=name, flags=flags, best_metric=best, best_epoch=best_ep,
                parameters=count_params(G)+count_params(D)+count_params(Q),
                training_time=tt, history_csv=hist_csv, ckpt_dir=ck)

# build the shared metric cache ONCE (fair across ablations), then run baseline A
build_metric_cache()
RESULTS={}
RESULTS["A_baseline"]=run_experiment("A_baseline", FLAG_TABLE["A_baseline"], resume=RESUME)

Metric cache built (train/val features, novelty ref, class centroids).
SEED = 42
Starting A_baseline from epoch 0
A_baseline
EPOCH 1 / 60

TRAIN
Generator Loss:       2.2701
Discriminator Loss:   0.5129
GAN Loss:             2.2701
Aesthetic Loss:       0.0000
Disentanglement Loss: 0.0000
Seam Loss:            0.0000
Novelty Loss:         0.0000
FID:                 402.0927
KID:                 0.4073
LPIPS:               0.8588
diversity:           3.6661
novelty:             0.4024
aesthetic_score:     0.5771
seamless_score:      0.8985
condition_adherence: nan
MMD:                 0.4649

VALIDATION
Generator Loss:       0.7549
Discriminator Loss:   0.5910
GAN Loss:             0.7549
Aesthetic Loss:       0.0000
Disentanglement Loss: 0.0000
Seam Loss:            0.0000
Novelty Loss:         0.0000
FID:                 405.7516
KID:                 0.4119
LPIPS:               0.7890
diversity:           3.7163
novelty:             0.4023
aesthetic_score:     0.5772
seamless_score: 

## 37 — Ablation B (+ Multi-Attribute Conditioning)

In [45]:
RESULTS["B_conditioning"]=run_experiment("B_conditioning", FLAG_TABLE["B_conditioning"], resume=RESUME)

SEED = 42
Starting B_conditioning from epoch 0
B_conditioning
EPOCH 1 / 60

TRAIN
Generator Loss:       5.4493
Discriminator Loss:   0.6535
GAN Loss:             5.4493
Aesthetic Loss:       0.0000
Disentanglement Loss: 0.0000
Seam Loss:            0.0000
Novelty Loss:         0.0000
FID:                 365.4089
KID:                 0.3275
LPIPS:               1.0317
diversity:           2.5914
novelty:             0.3965
aesthetic_score:     0.5866
seamless_score:      0.7991
condition_adherence: 0.5042
MMD:                 0.4380

VALIDATION
Generator Loss:       6.8521
Discriminator Loss:   1.1560
GAN Loss:             6.8521
Aesthetic Loss:       0.0000
Disentanglement Loss: 0.0000
Seam Loss:            0.0000
Novelty Loss:         0.0000
FID:                 381.6909
KID:                 0.3516
LPIPS:               0.9386
diversity:           2.5737
novelty:             0.3964
aesthetic_score:     0.5868
seamless_score:      0.7989
condition_adherence: 0.5123
MMD:                

## 38 — Ablation C (+ Disentangled Latent Space)

In [46]:
RESULTS["C_disentangled"]=run_experiment("C_disentangled", FLAG_TABLE["C_disentangled"], resume=RESUME)

SEED = 42
Starting C_disentangled from epoch 0
C_disentangled
EPOCH 1 / 60

TRAIN
Generator Loss:       6.8151
Discriminator Loss:   1.7218
GAN Loss:             5.8130
Aesthetic Loss:       0.0000
Disentanglement Loss: 1.0021
Seam Loss:            0.0000
Novelty Loss:         0.0000
FID:                 414.4071
KID:                 0.4247
LPIPS:               0.8960
diversity:           3.3200
novelty:             0.4366
aesthetic_score:     0.6069
seamless_score:      0.8175
condition_adherence: 0.5114
MMD:                 0.4912

VALIDATION
Generator Loss:       6.0048
Discriminator Loss:   0.9919
GAN Loss:             5.0033
Aesthetic Loss:       0.0000
Disentanglement Loss: 1.0015
Seam Loss:            0.0000
Novelty Loss:         0.0000
FID:                 425.5217
KID:                 0.4404
LPIPS:               0.8046
diversity:           3.2815
novelty:             0.4362
aesthetic_score:     0.6075
seamless_score:      0.8173
condition_adherence: 0.5058
MMD:                

## 39 — Ablation D (+ Seamless / Repeat-Aware Loss)

In [47]:
RESULTS["D_seamless"]=run_experiment("D_seamless", FLAG_TABLE["D_seamless"], resume=RESUME)

SEED = 42
Starting D_seamless from epoch 0
D_seamless
EPOCH 1 / 60

TRAIN
Generator Loss:       7.4620
Discriminator Loss:   1.6952
GAN Loss:             6.3268
Aesthetic Loss:       0.0000
Disentanglement Loss: 1.0022
Seam Loss:            0.1330
Novelty Loss:         0.0000
FID:                 421.0664
KID:                 0.4367
LPIPS:               0.9776
diversity:           2.1035
novelty:             0.4157
aesthetic_score:     0.6283
seamless_score:      0.9506
condition_adherence: 0.5033
MMD:                 0.4996

VALIDATION
Generator Loss:       -2.1749
Discriminator Loss:   4.3646
GAN Loss:             -3.2750
Aesthetic Loss:       0.0000
Disentanglement Loss: 1.0016
Seam Loss:            0.0985
Novelty Loss:         0.0000
FID:                 435.4958
KID:                 0.4575
LPIPS:               0.8771
diversity:           2.0831
novelty:             0.4162
aesthetic_score:     0.6289
seamless_score:      0.9506
condition_adherence: 0.4977
MMD:                 0.488

## 40 — Ablation E (+ Aesthetic Optimization)

In [48]:
RESULTS["E_aesthetic"]=run_experiment("E_aesthetic", FLAG_TABLE["E_aesthetic"], resume=RESUME)

SEED = 42
Starting E_aesthetic from epoch 0
E_aesthetic
EPOCH 1 / 60

TRAIN
Generator Loss:       7.3375
Discriminator Loss:   1.7180
GAN Loss:             6.0319
Aesthetic Loss:       0.3478
Disentanglement Loss: 1.0021
Seam Loss:            0.1295
Novelty Loss:         0.0000
FID:                 417.5312
KID:                 0.4265
LPIPS:               1.0121
diversity:           2.1539
novelty:             0.3921
aesthetic_score:     0.6104
seamless_score:      0.9537
condition_adherence: 0.5103
MMD:                 0.4977

VALIDATION
Generator Loss:       2.8731
Discriminator Loss:   0.3382
GAN Loss:             1.5843
Aesthetic Loss:       0.3896
Disentanglement Loss: 1.0015
Seam Loss:            0.0926
Novelty Loss:         0.0000
FID:                 436.8032
KID:                 0.4547
LPIPS:               0.9248
diversity:           2.1402
novelty:             0.3916
aesthetic_score:     0.6109
seamless_score:      0.9536
condition_adherence: 0.5047
MMD:                 0.487

## 41 — **Full ACDA-PatternGAN** (F: + Novelty optimization = all four novelties)

In [49]:
RESULTS["F_full_ACDA"]=run_experiment("F_full_ACDA", FLAG_TABLE["F_full_ACDA"], resume=RESUME)

SEED = 42
Starting F_full_ACDA from epoch 0
F_full_ACDA
EPOCH 1 / 60

TRAIN
Generator Loss:       7.6654
Discriminator Loss:   1.7009
GAN Loss:             6.1687
Aesthetic Loss:       0.3492
Disentanglement Loss: 1.0021
Seam Loss:            0.1250
Novelty Loss:         0.9747
FID:                 423.4210
KID:                 0.4335
LPIPS:               0.9993
diversity:           2.4884
novelty:             0.4607
aesthetic_score:     0.6001
seamless_score:      0.9511
condition_adherence: 0.5139
MMD:                 0.5025

VALIDATION
Generator Loss:       4.9447
Discriminator Loss:   0.4143
GAN Loss:             3.4465
Aesthetic Loss:       0.3998
Disentanglement Loss: 1.0016
Seam Loss:            0.0976
Novelty Loss:         0.9955
FID:                 436.8395
KID:                 0.4541
LPIPS:               0.9322
diversity:           2.5452
novelty:             0.4611
aesthetic_score:     0.6005
seamless_score:      0.9510
condition_adherence: 0.5084
MMD:                 0.487

## 42 — Final TEST evaluation (best checkpoints only) — reporting ONLY
Builds the TEST metric cache (first time TEST is touched), evaluates every model's **best** checkpoint
with the full protocol, saves `final_test_metrics.{csv,json}`.

In [50]:

def build_split_cache(split, df, n=None):
    n=n or CFG["eval_subset"]
    idx=np.random.default_rng(CFG["seed"]).choice(len(df), size=min(n,len(df)), replace=False)
    ld=DataLoader(BatikDataset(df.iloc[idx].reset_index(drop=True)), batch_size=64, shuffle=False)
    feats=features_from_loader(ld); imgs=torch.cat([xb for xb,_ in ld],0)
    METRIC_CACHE[split+"_feats"]=feats; METRIC_CACHE[split+"_mu_sigma"]=_mu_sigma(feats)
    METRIC_CACHE[split+"_real_imgs"]=imgs; METRIC_CACHE[split+"_cond"]=condition_pool(df, n)
build_split_cache("test", test_df, n=min(CFG["final_n_gen"], len(test_df)))

def load_for_eval(name, flags):
    G,D,Q=build_models(flags); optG,optD=build_optimizers(G,D,Q); schG,schD=build_schedulers(optG,optD)
    scaler=torch.cuda.amp.GradScaler(enabled=False); bundle=[G,D,Q,optG,optD,schG,schD,scaler,flags]
    best=os.path.join(CHECKPOINT_ROOT,name,"best_checkpoint.pt")
    p=best if os.path.isfile(best) else os.path.join(CHECKPOINT_ROOT,name,"latest_checkpoint.pt")
    load_ckpt(p, bundle); return bundle

TEST_METRICS={}
for name,res in RESULTS.items():
    b=load_for_eval(name, res["flags"])
    m=eval_metrics(b, split="test", n_gen=min(CFG["final_n_gen"], len(test_df)))
    m["parameters"]=res["parameters"]; m["training_time"]=res["training_time"]; m["best_epoch"]=res["best_epoch"]
    TEST_METRICS[name]=m
    print("TEST %-16s FID=%.2f KID=%.4f LPIPS=%.3f Div=%.2f Nov=%.3f Aes=%.3f Seam=%.3f Adh=%s MMD=%.4f"%(
        name, m["FID"], m["KID"], m["LPIPS"], m["diversity"], m["novelty"], m["aesthetic_score"],
        m["seamless_score"], ("%.3f"%m["condition_adherence"] if m["condition_adherence"]==m["condition_adherence"] else "n/a"), m["MMD"]))
fr=os.path.join(OUTPUT_ROOT,"final_results")
json.dump(TEST_METRICS, open(os.path.join(fr,"final_test_metrics.json"),"w"), indent=2, default=str)
pd.DataFrame(TEST_METRICS).T.to_csv(os.path.join(fr,"final_test_metrics.csv"))
print("Saved final_test_metrics.{json,csv}")

RNG restore warning: RNG state must be a torch.ByteTensor
TEST A_baseline       FID=203.36 KID=0.1014 LPIPS=0.577 Div=15.96 Nov=0.322 Aes=0.444 Seam=0.877 Adh=n/a MMD=0.1069
RNG restore warning: RNG state must be a torch.ByteTensor
TEST B_conditioning   FID=278.10 KID=0.1908 LPIPS=0.776 Div=11.78 Nov=0.381 Aes=0.501 Seam=0.850 Adh=0.540 MMD=0.2275
RNG restore warning: RNG state must be a torch.ByteTensor
TEST C_disentangled   FID=230.33 KID=0.1330 LPIPS=0.596 Div=15.52 Nov=0.341 Aes=0.440 Seam=0.830 Adh=0.815 MMD=0.1397
RNG restore warning: RNG state must be a torch.ByteTensor
TEST D_seamless       FID=228.99 KID=0.1311 LPIPS=0.567 Div=14.78 Nov=0.336 Aes=0.438 Seam=0.912 Adh=0.774 MMD=0.1404
RNG restore warning: RNG state must be a torch.ByteTensor
TEST E_aesthetic      FID=246.41 KID=0.1446 LPIPS=0.578 Div=14.85 Nov=0.359 Aes=0.449 Seam=0.923 Adh=0.721 MMD=0.1561
RNG restore warning: RNG state must be a torch.ByteTensor
TEST F_full_ACDA      FID=236.73 KID=0.1429 LPIPS=0.597 Div=14.2

## 43 — Latent disentanglement traversal (Full model) — qualitative, not proof

In [51]:

def traversal(name="F_full_ACDA", steps=8):
    res=RESULTS[name]; b=load_for_eval(name, res["flags"]); G=b[0]; flags=b[8]; G.eval()
    base_z=sample_z(1); cond=sample_cond(1, METRIC_CACHE["val_cond"]) if flags["cond"] else None
    tdir=os.path.join(OUTPUT_ROOT,"traversals"); os.makedirs(tdir, exist_ok=True)
    sens={}
    for sub,(a,c) in LATENT_SLICES.items():
        rows=[]; prev=None
        vals=torch.linspace(-2.5, 2.5, steps)
        for v in vals:
            z=base_z.clone(); z[:,a:c]=v
            with torch.no_grad(): img=G(z, cond)
            rows.append(img.cpu())
            if prev is not None: sens.setdefault(sub,[]).append(float((img-prev).pow(2).mean().sqrt()))
            prev=img
        grid=torch.cat(rows,0)
        save_grid(grid, os.path.join(tdir, "%s_traversal.png"%sub), "%s traversal (%s)"%(sub,name), nrow=steps)
    sensitivity={k: float(np.mean(v)) for k,v in sens.items()}
    json.dump(sensitivity, open(os.path.join(tdir,"latent_sensitivity.json"),"w"), indent=2)
    print("Latent sensitivity (mean output change per step):", sensitivity)
    print("NOTE: qualitative indicator only, not definitive proof of disentanglement.")
traversal()

RNG restore warning: RNG state must be a torch.ByteTensor
Latent sensitivity (mean output change per step): {'structure': 0.24832540111882345, 'style': 0.20476744323968887, 'color': 0.2059250431401389, 'texture': 0.1705214668597494}
NOTE: qualitative indicator only, not definitive proof of disentanglement.


## 44 — Ablation comparison table + CSV

In [52]:

ORDER=["A_baseline","B_conditioning","C_disentangled","D_seamless","E_aesthetic","F_full_ACDA"]
LABEL={"A_baseline":"Baseline GAN","B_conditioning":"+ Conditioning","C_disentangled":"+ Disentanglement",
       "D_seamless":"+ Seamless","E_aesthetic":"+ Aesthetic","F_full_ACDA":"Full ACDA-PatternGAN"}
rows=[]
for k in ORDER:
    if k not in TEST_METRICS: continue
    m=TEST_METRICS[k]
    rows.append(dict(Model=LABEL[k], FID=round(m["FID"],3), KID=round(m["KID"],5), LPIPS=round(m["LPIPS"],4),
        Diversity=round(m["diversity"],3), Novelty=round(m["novelty"],4),
        **{"Aesthetic Score":round(m["aesthetic_score"],4), "Seamless Score":round(m["seamless_score"],4),
           "Condition Adherence":(round(m["condition_adherence"],4) if m["condition_adherence"]==m["condition_adherence"] else "n/a")},
        MMD=round(m["MMD"],5), Parameters=int(m["parameters"]),
        **{"Training Time":round(m["training_time"],1), "Best Epoch":int(m["best_epoch"])}))
ablation_df=pd.DataFrame(rows)
ablation_df.to_csv(os.path.join(OUTPUT_ROOT,"final_results","ablation_comparison.csv"), index=False)
print(ablation_df.to_string(index=False))

               Model     FID     KID  LPIPS  Diversity  Novelty  Aesthetic Score  Seamless Score Condition Adherence     MMD  Parameters  Training Time  Best Epoch
        Baseline GAN 203.362 0.10142 0.5773     15.956   0.3215           0.4440          0.8771                 n/a 0.10685     6579460         1987.0          57
      + Conditioning 278.096 0.19079 0.7761     11.778   0.3807           0.5010          0.8501              0.5401 0.22752     7711236          950.8          14
   + Disentanglement 230.331 0.13296 0.5962     15.517   0.3411           0.4401          0.8300              0.8147 0.13967    11516484         1971.7          54
          + Seamless 228.992 0.13111 0.5674     14.780   0.3363           0.4380          0.9119              0.7739 0.14037    11516484         1933.4          45
         + Aesthetic 246.412 0.14465 0.5779     14.852   0.3587           0.4491          0.9233              0.7212 0.15614    11516484         1180.8          22
Full ACDA-Patter

## 45 — Final metrics aggregation (best VALIDATION + final TEST)

In [53]:

def best_val_row(name):
    df=pd.read_csv(RESULTS[name]["history_csv"])
    if "val_FID" not in df or not df["val_FID"].notna().any(): return {}
    r=df.loc[df["val_FID"].idxmin()]
    return {("best_"+c): (float(r[c]) if pd.api.types.is_number(r[c]) else r[c])
            for c in df.columns if c.startswith("val_")}
final={"dataset":"Batik Lasem",
       "train_images":int(len(train_df)),"val_images":int(len(val_df)),"test_images":int(len(test_df)),
       "num_motifs":int(NUM_MOTIFS),"motifs":MOTIFS,
       "full_model_best_validation":best_val_row("F_full_ACDA"),
       "final_test_metrics":TEST_METRICS,
       "targets":CFG["targets"]}
# target check (targets are goals, NOT results)
ft=TEST_METRICS.get("F_full_ACDA",{})
tgt=CFG["targets"]; checks={}
if ft:
    checks["FID<=target"]      = ("achieved" if ft["FID"]<=tgt["FID"] else "Target not achieved")
    checks["KID<=target"]      = ("achieved" if ft["KID"]<=tgt["KID"] else "Target not achieved")
    checks["Aesthetic>=target"]= ("achieved" if ft["aesthetic_score"]>=tgt["AESTHETIC"] else "Target not achieved")
    checks["Seamless>=target"] = ("achieved" if ft["seamless_score"]>=tgt["SEAMLESS"] else "Target not achieved")
    checks["Novelty>=target"]  = ("achieved" if ft["novelty"]>=tgt["NOVELTY"] else "Target not achieved")
final["target_checks"]=checks
json.dump(final, open(os.path.join(OUTPUT_ROOT,"final_results","final_report.json"),"w"), indent=2, default=str)
print(json.dumps({"sizes":dict(train=len(train_df),val=len(val_df),test=len(test_df)),
                  "target_checks":checks}, indent=2))

{
  "sizes": {
    "train": 3990,
    "val": 590,
    "test": 1280
  },
  "target_checks": {
    "FID<=target": "Target not achieved",
    "KID<=target": "Target not achieved",
    "Aesthetic>=target": "Target not achieved",
    "Seamless>=target": "achieved",
    "Novelty>=target": "achieved"
  }
}


## 46 — Publication plots (per-model histories + ablation comparison bars)

In [54]:

# per-model training-history plots
for k,res in RESULTS.items():
    df=pd.read_csv(res["history_csv"]); plot_history(df, os.path.join(OUTPUT_ROOT,"plots",k), LABEL.get(k,k))
# ablation comparison bar charts
def bar(metric, better):
    labels=[LABEL[k] for k in ORDER if k in TEST_METRICS]
    vals=[TEST_METRICS[k].get(metric,float("nan")) for k in ORDER if k in TEST_METRICS]
    fig,ax=plt.subplots(figsize=(11,6)); ax.bar(range(len(vals)), vals, color="#c0392b")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_title("%s (%s is better)"%(metric, better)); ax.set_ylabel(metric); ax.grid(axis="y",alpha=.3)
    for i,v in enumerate(vals):
        if v==v: ax.text(i, v, "%.3g"%v, ha="center", va="bottom")
    fig.tight_layout(); fig.savefig(os.path.join(OUTPUT_ROOT,"plots","ablation_%s.png"%metric), dpi=300, bbox_inches="tight"); plt.close(fig)
for mtr,dirn in [("FID","lower"),("KID","lower"),("LPIPS","lower"),("diversity","higher"),("novelty","higher"),
                 ("aesthetic_score","higher"),("seamless_score","higher"),("condition_adherence","higher"),("MMD","lower")]:
    bar(mtr, dirn)
print("Publication plots saved under", os.path.join(OUTPUT_ROOT,"plots"))

Publication plots saved under /content/drive/MyDrive/hari/Textile_Pattern_GAN/acda_outputs/plots


## 47 — Final summary + dataset-calibrated quality classification
Labels (Strong/Good/Moderate/Weak) are computed from ACTUAL results relative to a **real-vs-real**
reference and the **baseline**, using documented rules — never hard-coded to favour the proposed model.

In [55]:

# real-vs-real references (context; split a real feature set in half)
def real_real_refs():
    f=METRIC_CACHE["test_feats"]; n=f.shape[0]//2
    a,b=f[:n], f[n:2*n]
    mu_a,sig_a=_mu_sigma(a)
    return dict(REAL_REAL_FID=fid_from_feats(b, mu_a, sig_a),
                REAL_REAL_MMD=mmd_rbf(a,b)["mmd"],
                REAL_REAL_DIVERSITY=diversity_from_feats(f)["diversity_mean"])
REFS=real_real_refs()
base=TEST_METRICS["A_baseline"]; full=TEST_METRICS["F_full_ACDA"]
def better(metric, lower=True):
    bv, fv = base[metric], full[metric]
    return (fv < bv) if lower else (fv > bv)
improvements=dict(
    FID=better("FID",True), KID=better("KID",True), MMD=better("MMD",True), LPIPS=better("LPIPS",True),
    diversity=better("diversity",False), novelty=better("novelty",False),
    aesthetic_score=better("aesthetic_score",False), seamless_score=better("seamless_score",False),
    condition_adherence=(full["condition_adherence"]==full["condition_adherence"] and base["condition_adherence"]!=base["condition_adherence"]) or
                        (full["condition_adherence"]==full["condition_adherence"] and base["condition_adherence"]==base["condition_adherence"] and full["condition_adherence"]>base["condition_adherence"]))
n_better=sum(1 for v in improvements.values() if v)
# closeness of full FID to the real-real floor, relative to baseline gap
fid_gain = (base["FID"]-full["FID"])/max(base["FID"]-REFS["REAL_REAL_FID"],1e-6)
if   n_better>=7 and full["FID"]<base["FID"] and fid_gain>=0.5: label="Strong"
elif n_better>=5 and full["FID"]<base["FID"]:                    label="Good"
elif n_better>=3:                                                label="Moderate"
else:                                                            label="Weak"
print("="*80); print("ACDA-PatternGAN — FINAL SUMMARY"); print("="*80)
print("Dataset: Batik Lasem | train=%d val=%d test=%d | motifs=%d"%(len(train_df),len(val_df),len(test_df),NUM_MOTIFS))
print("Real-vs-real references:", {k:round(v,4) for k,v in REFS.items()})
print("\nBaseline vs Full (TEST):")
for m in NINE:
    bv=base[m]; fv=full[m]
    print("  %-20s baseline=%-9s full=%-9s"%(m, ("%.4f"%bv if bv==bv else "n/a"), ("%.4f"%fv if fv==fv else "n/a")))
print("\nImprovements over baseline:", {k:bool(v) for k,v in improvements.items()})
print("Metrics improved: %d/9 | FID gain toward real-real floor: %.2f"%(n_better, fid_gain))
print("\nQUALITY CLASSIFICATION (data-calibrated, rule-based): **%s**"%label)
print("Rules: Strong = >=7/9 improved AND FID gain>=0.5 toward real-real floor;")
print("       Good = >=5/9 improved AND FID<baseline; Moderate = >=3/9; else Weak.")
print("Targets are goals (see target_checks) — 'Target not achieved' reported honestly where applicable.")
print("="*80)
json.dump(dict(quality_label=label, improvements={k:bool(v) for k,v in improvements.items()},
               n_better=n_better, fid_gain_toward_real=fid_gain, real_real_refs=REFS),
          open(os.path.join(OUTPUT_ROOT,"final_results","quality_classification.json"),"w"), indent=2, default=str)
print("Done. All artifacts under:", OUTPUT_ROOT)

ACDA-PatternGAN — FINAL SUMMARY
Dataset: Batik Lasem | train=3990 val=590 test=1280 | motifs=5
Real-vs-real references: {'REAL_REAL_FID': 39.6356, 'REAL_REAL_MMD': 0.001, 'REAL_REAL_DIVERSITY': 19.1072}

Baseline vs Full (TEST):
  FID                  baseline=203.3623  full=236.7304 
  KID                  baseline=0.1014    full=0.1429   
  LPIPS                baseline=0.5773    full=0.5972   
  diversity            baseline=15.9561   full=14.2408  
  novelty              baseline=0.3215    full=0.3483   
  aesthetic_score      baseline=0.4440    full=0.4317   
  seamless_score       baseline=0.8771    full=0.9115   
  condition_adherence  baseline=n/a       full=0.7993   
  MMD                  baseline=0.1069    full=0.1596   

Improvements over baseline: {'FID': False, 'KID': False, 'MMD': False, 'LPIPS': False, 'diversity': False, 'novelty': True, 'aesthetic_score': False, 'seamless_score': True, 'condition_adherence': True}
Metrics improved: 3/9 | FID gain toward real-real floo